In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
#from pmdarima.arima import auto_arima
from math import sqrt
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima_model import ARIMA
from statsmodels.tsa.stattools import adfuller
import statsmodels.tsa.stattools as tsa
from numpy import log

In [ ]:
# ORTHO-EAST

df = pd.read_csv("data/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df

In [ ]:
# Definir os limites Alqueva
norte_min = 1855050
norte_max = 1855850
este_min = 2792250
este_max = 2793250

# Aplicar o filtro
df = df[
    (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
    (df['easting'] >= este_min) & (df['easting'] <= este_max)
]

df

In [ ]:
s = df.T
s.head(25)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Criar GeoDataFrame com EPSG:3035
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # European LAEA projection
)

# Converter para Web Mercator para compatibilidade com o mapa de fundo
gdf_webmerc = gdf.to_crs(epsg=3857)

# Plot com mapa de fundo
fig, ax = plt.subplots(figsize=(10, 10))
gdf_webmerc.plot(ax=ax, color='red', markersize=50, label='Meus pontos')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
#ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()
plt.legend()
plt.title("Localização dos Pontos com Mapa de Fundo")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
import geopandas as gpd

# Criar o GeoDataFrame com o CRS correto (ETRS89 / LAEA Europe)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # ou outro se souberes que não é este
)

# Converter para Web Mercator para usar no mapa
gdf_webmerc = gdf.to_crs(epsg=3857)

# Escolher a coluna para o valor
coluna_valor = "mean_velocity"

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf[coluna_valor].min(),
    vcenter=0,
    vmax=gdf[coluna_valor].max()
)

# Colormap
cmap = plt.cm.get_cmap('jet')

# Criar figura
fig, ax = plt.subplots(figsize=(10, 10))
gdf_webmerc.plot(
    ax=ax,
    column=coluna_valor,
    cmap=cmap,
    markersize=30,
    norm=norm,
    legend=False
)

# Adicionar mapa de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title(f"Pontos com escala de cores ({coluna_valor})")

# Adicionar barra de cores
from matplotlib.cm import ScalarMappable
sm = ScalarMappable(cmap=cmap, norm=norm)
sm._A = []  # Forçar ScalarMappable a funcionar
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(coluna_valor)

plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
import geopandas as gpd

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Variáveis a representar
variaveis = [
    "height",
    "rmse",
    "mean_velocity",
    "mean_velocity_std",
    "acceleration",
    "acceleration_std",
    "seasonality",
    "seasonality_std"
]

# Criar figura 4x2 (mais altura, mais espaço)
fig, axs = plt.subplots(2, 4, figsize=(20, 10), constrained_layout=True)
axs = axs.flatten()

for i, var in enumerate(variaveis):
    ax = axs[i]

    vmin = gdf_webmerc[var].min()
    vmax = gdf_webmerc[var].max()

    # Normalização de cores
    if vmin < 0 and vmax > 0:
        norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    else:
        norm = colors.Normalize(vmin=vmin, vmax=vmax)

    # Plot principal
    gdf_webmerc.plot(
        ax=ax,
        column=var,
        cmap='jet',
        markersize=10,
        norm=norm,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(var, fontsize=10)
    ax.set_axis_off()

    # Colorbar individual
    sm = plt.cm.ScalarMappable(cmap='jet', norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.01)
    cbar.ax.tick_params(labelsize=7)

# Exibir
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Nome da variável e PID de interesse
variavel_cor = "mean_velocity"
pid_escolhido = '40McjqvU5C'  # Substitui se quiseres outro

# Isolar colunas temporais
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# Verifica se o PID existe
linha_ponto = df[df["pid"] == pid_escolhido]
linha_ponto_geo = gdf_webmerc[gdf_webmerc["pid"] == pid_escolhido]

if linha_ponto.empty:
    print(f"PID {pid_escolhido} não encontrado!")
else:
    serie_temporal = linha_ponto[colunas_temporais].values.flatten().astype(float)

    # Normalização das cores
    norm = colors.TwoSlopeNorm(
        vmin=gdf_webmerc[variavel_cor].min(),
        vcenter=0,
        vmax=gdf_webmerc[variavel_cor].max()
    )

    # --- Criar figura com layout ajustado ---
    fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

    # --- Painel do mapa ---
    divider = make_axes_locatable(axs[0])
    cax = divider.append_axes("right", size="5%", pad=0.1)

    gdf_webmerc.plot(
        ax=axs[0],
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        legend=True,
        norm=norm,
        cax=cax
    )
    ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
    axs[0].set_title(f"Mapa com escala de {variavel_cor}")
    axs[0].set_axis_off()

    # Destacar ponto específico com círculo maior
    linha_ponto_geo.plot(
        ax=axs[0],
        color='none',
        edgecolor='black',
        linewidth=2,
        markersize=200,
        zorder=3
    )
    axs[0].scatter(
        linha_ponto_geo.geometry.x,
        linha_ponto_geo.geometry.y,
        color='yellow',
        s=80,
        zorder=4,
        edgecolor='black'
    )

    # --- Painel da série temporal ---
    axs[1].plot(datas, serie_temporal, marker='o', linestyle='-', color='blue')
    axs[1].set_title(f"Série temporal do deslocamento\nPID {pid_escolhido}")
    axs[1].set_xlabel("Data")
    axs[1].set_ylabel("Deslocamento (mm)")
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# --- Variáveis principais ---
variavel_cor = "mean_velocity"
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Selecionar os 10 pontos com maior mean_velocity ---
top10 = gdf_webmerc.sort_values(by=variavel_cor, ascending=False).head(10)

# --- Normalização da cor ---
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura ---
fig, axs = plt.subplots(nrows=10, ncols=2, figsize=(14, 40), gridspec_kw={'width_ratios': [1.2, 1]})

for i, (idx, linha) in enumerate(top10.iterrows()):
    ax_map = axs[i, 0]
    ax_plot = axs[i, 1]

    # --- Mapa ---
    divider = make_axes_locatable(ax_map)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    gdf_webmerc.plot(
        ax=ax_map,
        column=variavel_cor,
        cmap='jet',
        markersize=20,
        norm=norm,
        cax=cax,
        legend=True
    )
    ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
    ax_map.set_axis_off()
    ax_map.set_title(f"Mapa do PID {linha['pid']}")

    # Destacar o ponto
    ax_map.scatter(
        linha.geometry.x,
        linha.geometry.y,
        color='yellow',
        s=200,
        edgecolor='black',
        linewidth=2,
        zorder=4
    )

    # --- Série temporal ---
    serie = df.loc[df['pid'] == linha['pid'], colunas_temporais].values.flatten().astype(float)
    ax_plot.plot(datas, serie, marker='o', linestyle='-', color='blue', label='Série')

    # --- Trend line ---
    x = (datas - datas[0]).days  # dias desde o início (como base temporal)
    coeffs = np.polyfit(x, serie, deg=2)  # ajuste linear
    trend = np.poly1d(coeffs)(x)
    ax_plot.plot(datas, trend, color='red', linestyle='--', label='Trend line')

    ax_plot.set_title(f"Série temporal - PID {linha['pid']}")
    ax_plot.set_xlabel("Data")
    ax_plot.set_ylabel("Deslocamento (mm)")
    ax_plot.grid(True)
    ax_plot.legend(fontsize=8)

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # CRS dos dados fornecidos
)
gdf_webmerc = gdf.to_crs(epsg=3857)  # Para compatibilidade com o basemap

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = 0.5

# Isolar colunas com datas (começam com "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Filtrar os pontos para destacar ---
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura com layout ajustado ---
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar pontos acima do limite ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    axs[1].plot(datas, serie, alpha=0.5)

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"  # CRS dos dados fornecidos
)
gdf_webmerc = gdf.to_crs(epsg=3857)  # Para compatibilidade com o basemap

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = -0.5

# Isolar colunas com datas (começam com "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# --- Filtrar os pontos para destacar ---
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização das cores
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# --- Criar figura com layout ajustado ---
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar todos os pontos acima do limite com círculos maiores ---
for idx, ponto in pontos_destaque.iterrows():
    x, y = ponto.geometry.x, ponto.geometry.y
    axs[0].scatter(x, y, color='yellow', s=400, edgecolor='black', linewidth=2)

# --- Destacar o ponto original (ponto com marcador menor) ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    axs[1].plot(datas, serie, alpha=0.5)

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

# Ajuste no layout para garantir que tudo caiba bem
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import contextily as ctx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Criação do GeoDataFrame com coordenadas easting/northing ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.easting, df.northing),
    crs="EPSG:3035"
)
gdf_webmerc = gdf.to_crs(epsg=3857)

# Nome da variável e limite
variavel_cor = "mean_velocity"
limite = -0.5

# Colunas temporais (as que começam por "20")
colunas_temporais = [col for col in df.columns if col.startswith("20")]
datas = pd.to_datetime(colunas_temporais)

# Filtrar os pontos a destacar
pontos_destacados = df[df[variavel_cor] > limite]
pontos_destaque = gdf_webmerc[gdf_webmerc[variavel_cor] > limite]

# Normalização da cor
norm = colors.TwoSlopeNorm(
    vmin=gdf_webmerc[variavel_cor].min(),
    vcenter=0,
    vmax=gdf_webmerc[variavel_cor].max()
)

# Criar figura
fig, axs = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [1.2, 1]})

# --- Painel do mapa ---
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.1)

gdf_webmerc.plot(
    ax=axs[0],
    column=variavel_cor,
    cmap='jet',
    markersize=20,
    legend=True,
    norm=norm,
    cax=cax
)
ctx.add_basemap(axs[0], source=ctx.providers.Esri.WorldImagery)
axs[0].set_title(f"Mapa com escala de {variavel_cor}")
axs[0].set_axis_off()

# --- Destacar os pontos com círculos e legendas com PID ---
for idx, ponto in pontos_destaque.iterrows():
    x, y = ponto.geometry.x, ponto.geometry.y
    axs[0].scatter(x, y, color='yellow', s=400, edgecolor='black', linewidth=2)
    axs[0].annotate(
        texto := ponto["pid"],
        (x, y),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=7,
        color='white',
        weight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6)
    )

# --- Reforçar os pontos destacados em vermelho ---
pontos_destaque.plot(
    ax=axs[0],
    facecolor='red',
    edgecolor='black',
    markersize=80,
    linewidth=1,
    label=f"{variavel_cor} > {limite} mm/ano"
)

# --- Painel da série temporal ---
for idx, row in pontos_destacados.iterrows():
    serie = row[colunas_temporais].values.astype(float)
    pid = row["pid"]
    axs[1].plot(datas, serie, alpha=0.6, label=pid)  # Adiciona label com o PID

axs[1].set_title(f"Séries temporais dos pontos com {variavel_cor} > {limite} mm/ano")
axs[1].set_xlabel("Data")
axs[1].set_ylabel("Deslocamento (mm)")
axs[1].grid(True)

# Mostrar legenda (com número limitado para não sobrecarregar)
axs[1].legend(loc="best", fontsize=8, ncol=2, title="PID")

plt.tight_layout()
plt.show()


In [ ]:
import folium
import pandas as pd

# Dados dos teus pontos
#df = pd.read_csv("data/meus_pontos.csv")

# Centro do mapa
mapa = folium.Map(location=[df['easting'].mean(), df['northing'].mean()], zoom_start=12)

# Adiciona cada ponto
for _, row in df.iterrows():
    folium.Marker(
        location=[row['easting'], row['northing']],
        popup=f"Ponto: {row['nome'] if 'nome' in row else ''}"
    ).add_to(mapa)

# Mostrar (num Jupyter Notebook aparece diretamente)
mapa.save("meu_mapa.html")

In [ ]:
import folium
from folium.plugins import MarkerCluster

# Converter para lat/lon para o mapa (Folium exige EPSG:4326)
gdf_latlon = gdf.to_crs(epsg=4326)

# Criar mapa centrado nos dados
centro = [gdf_latlon.geometry.y.mean(), gdf_latlon.geometry.x.mean()]
m = folium.Map(location=centro, zoom_start=10, tiles='Esri.WorldImagery')

# Cluster para os pontos
marker_cluster = MarkerCluster().add_to(m)

# Colormap simples (poderíamos fazer algo mais customizado com branca ou branca-colormap)
for idx, row in gdf_latlon.iterrows():
    valor = row[variavel_cor]
    cor = 'red' if valor > limite else 'blue'
    popup_text = f"PID: {row['pid']}<br>{variavel_cor}: {valor:.2f} mm/ano"
    
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color='black',
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200)
    ).add_to(marker_cluster)

# Mostrar
m.save("mapa_folium.html")
